# MODELO NEURALPROPHET PARA EL CONTAMINANTE CO PARA BARCELONA

Importamos las librerías y definimos las rutas.

In [13]:
from pathlib import Path
from sklearn.model_selection import ParameterGrid

import warnings
warnings.filterwarnings("ignore")

import logging
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)
logging.getLogger("fbprophet").setLevel(logging.WARNING)

import matplotlib.pyplot as plt
from pylab import rcParams
plt.style.use("fivethirtyeight")
plt.rcParams["lines.linewidth"] = 1.5
light_style = {
    "figure.facecolor": "#d9effb",   
    "axes.facecolor": "#d9effb",
    "savefig.facecolor": "#d9effb",
    "axes.grid": True,
    "axes.grid.which": "both",
    "axes.spines.left": True,
    "axes.spines.right": True,
    "axes.spines.top": True,
    "axes.spines.bottom": True,
    "grid.color": "#a9d3f2",
    "grid.linewidth": "0.8",
    "text.color": "#333333",
    "axes.labelcolor": "#333333",
    "axes.labelweight": "black",      
    "xtick.color": "#333333",
    "ytick.color": "#333333",
    "font.size": 12,
    "axes.titleweight": "bold",      
    "legend.fontsize": 12,
    "legend.title_fontsize": 12,
}
plt.rcParams.update(light_style)
rcParams["figure.figsize"] = (18, 7)


import sys
import importlib
from pathlib import Path

SCRIPTS_PATH = Path.cwd().parents[2]

if str(SCRIPTS_PATH) not in sys.path:
    sys.path.append(str(SCRIPTS_PATH))

import utils

importlib.reload(utils)

from utils import CARGA_Y_FILTRO, EVALUAR_METRICAS, BUSQUEDA_CONFIGURACIONES_NEURALPROPHET, ENTRENAR_EVALUAR_NEURALPROPHET

In [14]:
BASE_PATH = Path("..", "..", "..", "..")
FOLDER_DATA = BASE_PATH / "datasets" / "eda_archivos_cont_clima_indices"

Cargamos los datos y filtramos las columnas que realmente necesitamos y como ciudad elegimos únicamente Barcelona.

In [15]:
df1 = CARGA_Y_FILTRO(
    contaminante="CO (mg.m-3)",
    ciudad="Barcelona"
)

## División del conjunto de datos y creación de los regresores futuros y retardados

In [16]:
# ==============================================================================
# División cronológica del conjunto de datos
# ==============================================================================

train = df1[df1["Año"] <= 2020].copy()

validation = df1[
    (df1["Año"] >= 2021) &
    (df1["Año"] <= 2022)
].copy()

test = df1[df1["Año"] >= 2023].copy()

print(f"Entrenamiento: {train['Start'].min()} -> {train['Start'].max()}")
print(f"Validación:    {validation['Start'].min()} -> {validation['Start'].max()}")
print(f"Prueba:        {test['Start'].min()} -> {test['Start'].max()}")

print()
print(f"Nº muestras entrenamiento: {len(train):,}")
print(f"Nº muestras validación:    {len(validation):,}")
print(f"Nº muestras prueba:        {len(test):,}")

Entrenamiento: 2013-01-01 01:00:00 -> 2020-12-31 23:00:00
Validación:    2021-01-01 00:00:00 -> 2022-12-31 23:00:00
Prueba:        2023-01-01 00:00:00 -> 2024-12-31 23:00:00

Nº muestras entrenamiento: 70,124
Nº muestras validación:    17,520
Nº muestras prueba:        17,544


NeuralProphet espera que los datos estén formateados de una manera específica. El modelo requiere una columna *ds* que contenga las fechas y una columna *y* que contenga los valores que queremos modelar/predecir.

In [17]:
train = train.rename(columns={"Start": "ds", "CO (mg.m-3)": "y"})
validation = validation.rename(columns={"Start": "ds", "CO (mg.m-3)": "y"})
test = test.rename(columns={"Start": "ds", "CO (mg.m-3)": "y"})

In [18]:
variables_exogenas= [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "surface_pressure",
    "cloudcover",
    "windspeed_10m",
    "shortwave_radiation",
    "boundary_layer_height",
    "NDVI",
    "NDBI"
]

Nos quedamos exclusivamente con las variables necesarias para el modelado.

In [19]:
# Lista de columnas que quieres mantener
columnas = ['ds', 'y'] + variables_exogenas

train = train[columnas]
validation = validation[columnas]

Definimos los regresores retardados y los regresores futuros.

In [20]:
regresores_retardados = variables_exogenas.copy()
regresores_retardados.remove("NDBI")
regresores_retardados.remove("NDVI")

regresores_futuros = ["NDVI", "NDBI"]

## Selección de los mejores hiperparámetros

### Definición del espacio de búsqueda

In [ ]:
# ==============================================================================
# Valores candidatos de los hiperparámetros
# ==============================================================================

param_grid = {

    # --------------------------------------------------------------------------
    # Tendencia
    # --------------------------------------------------------------------------

    # Parte inicial de la serie en la que se colocan los puntos de cambio
    "changepoints_range": [
        0.8,
        0.9
    ],

    # Regularización de los cambios de tendencia
    "trend_reg": [
        0.1,
        1.0
    ],


    # --------------------------------------------------------------------------
    # Estacionalidad
    # --------------------------------------------------------------------------

    # Tipo de estacionalidad
    "seasonality_mode": [
        "multiplicative"
    ],

    # Regularización de las componentes estacionales
    "seasonality_reg": [
        0.1,
        1.0
    ],

    # Número de términos de Fourier de la estacionalidad anual
    "yearly_seasonality": [
        10,
        20
    ],

    # Número de términos de Fourier de la estacionalidad semanal
    "weekly_seasonality": [
        10,
        20
    ],

    # Número de términos de Fourier de la estacionalidad diaria
    "daily_seasonality": [
        10,
        20
    ],

    # --------------------------------------------------------------------------
    # Autorregresión
    # --------------------------------------------------------------------------

    # Número de valores anteriores de CO utilizados por AR-Net
    "n_lags": [
        48,
        168
    ],

    # Capas ocultas de la red autorregresiva
    "ar_layers": [
        [],
        [32],
        [64, 32]
    ],

    # Capas ocultas para los regresores retardados
    "lagged_reg_layers": [
        [],
        [32],
        [64, 32]
    ],

    # --------------------------------------------------------------------------
    # Entrenamiento
    # --------------------------------------------------------------------------

    # Número de épocas
    "epochs": [
        50,
        100
    ],

    # Tamaño de los lotes
    "batch_size": [
        64,
        128,
        256
    ],

    # --------------------------------------------------------------------------
    # Festivos
    # --------------------------------------------------------------------------

    # Incorporación de los festivos nacionales de España
    "usar_festivos": [
        True
    ],
}

### Búsqueda del mejor modelo

In [22]:
# ==============================================================================
# Generación de todas las combinaciones posibles
# ==============================================================================

configuraciones = list(
    ParameterGrid(param_grid)
)

print(
    f"Número total de configuraciones que se evaluarán: "
    f"{len(configuraciones)}"
)

Número total de configuraciones que se evaluarán: 6912


In [ ]:
(
    resultados_df,
    resultados_correctos,
    mejor_resultado,
    mejor_modelo
) = BUSQUEDA_CONFIGURACIONES_NEURALPROPHET(
    configuraciones=configuraciones,
    train=train,
    validation=validation,
    estacionalidades=[
        "yearly",
        "weekly",
        "daily"
    ],
    regresores_futuros=regresores_futuros,
    regresores_retardados=regresores_retardados
)

Estacionalidades utilizadas:
- yearly
- weekly
- daily

Mejores parámetros:

changepoints_range: 0.9
trend_reg: 1.0
seasonality_mode: multiplicative
seasonality_reg: 1.0
yearly_seasonality: 20
weekly_seasonality: 10
daily_seasonality: 10
n_lags: 168
ar_layers: [64, 32]
lagged_reg_layers: [64, 32]
epochs: 50
batch_size: 256
usar_festivos: True

MSE de validación: 0.007341


In [24]:
# Mejores parámetros obtenidos tras la búsqueda de hiperparámetros

mejores_parametros = {
    "changepoints_range": 0.9,
    "trend_reg": 1.0,
    "seasonality_mode": "multiplicative",
    "seasonality_reg": 1.0,
    "yearly_seasonality": 20,
    "weekly_seasonality": 10,
    "daily_seasonality": 10,
    "n_lags": 168,
    "ar_layers": [64, 32],
    "lagged_reg_layers": [64, 32],
    "epochs": 50,
    "batch_size": 256,
    "usar_festivos": True,
}

In [ ]:
(
    modelo_neuralprophet,
    historial_entrenamiento,
    prediccion_validacion,
    comparacion_validacion,
    metricas_validacion,
    train_neuralprophet_final,
    validation_neuralprophet_final
) = ENTRENAR_EVALUAR_NEURALPROPHET(
    train=train,
    validation=validation,
    regresores_futuros=regresores_futuros,
    regresores_retardados=regresores_retardados,
    mejores_parametros=mejores_parametros
)

Resultados de la evaluación del modelo
--------------------------------------
Error absoluto medio (MAE): 0.0514791
Error cuadrático medio (MSE): 0.007341
Raíz del error cuadrático medio (RMSE): 0.08567
Error porcentual absoluto medio (MAPE): 15.47 %
Raíz del error cuadrático medio normalizada (NRMSE): 27.79 %
